In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available (speeds up training significantly)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.10.0+cpu


In [2]:
OUTPUT_DIR = Path(r'C:\Users\taylo\hyperwind_now\data\processed')
MODEL_DIR  = Path(r'C:\Users\taylo\hyperwind_now\models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Sequence config (mirrors TrajGRU from Doc 8)
IN_LEN    = 5   # use 5 timesteps as input (15 hours at 3h resolution)
OUT_LEN   = 4   # predict next 4 timesteps (12 hours ahead)
N_FEATURES = 66 # from Module 1 combined dataset

# Training config
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 0.001
PATIENCE    = 10  # early stopping

# Train/val/test periods (temporal split - Doc 11 pattern)
TRAIN_END = "2021-12-31"
VAL_START = "2022-01-01"
VAL_END   = "2022-12-31"
TEST_START = "2023-01-01"

print("Config ready!")
print(f"Input sequence  : {IN_LEN} timesteps")
print(f"Forecast horizon: {OUT_LEN} timesteps ({OUT_LEN*3} hours ahead)")
print(f"Features        : {N_FEATURES}")

Config ready!
Input sequence  : 5 timesteps
Forecast horizon: 4 timesteps (12 hours ahead)
Features        : 66


In [3]:
def load_all_monthly_data():
    """
    Load all processed monthly CSV files from Module 1
    and concatenate into one large DataFrame.
    """
    all_files = sorted(OUTPUT_DIR.glob("hyperwind_module1_*.csv"))
    
    if len(all_files) == 0:
        raise FileNotFoundError(
            f"No processed files found in {OUTPUT_DIR}. "
            "Run Module 1 first!"
        )
    
    dfs = []
    for f in all_files:
        df = pd.read_csv(f, index_col=0, parse_dates=True)
        dfs.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    
    df_all = pd.concat(dfs).sort_index()
    df_all = df_all[~df_all.index.duplicated(keep='first')]
    
    print(f"\nTotal: {len(df_all)} timesteps")
    print(f"Period: {df_all.index.min()} to {df_all.index.max()}")
    print(f"Features: {df_all.shape[1]}")
    
    return df_all

df_all = load_all_monthly_data()

Loaded hyperwind_module1_jan2020.csv: 240 rows

Total: 240 timesteps
Period: 2020-01-01 00:00:00 to 2020-01-30 21:00:00
Features: 66


In [4]:
def prepare_data(df):
    """
    Normalize features and split into train/val/test.
    Uses temporal split (Doc 11 pattern) not random split.
    """
    # Temporal split
    df_train = df[df.index <= TRAIN_END]
    df_val   = df[(df.index >= VAL_START) & (df.index <= VAL_END)]
    df_test  = df[df.index >= TEST_START]
    
    print(f"Train: {len(df_train)} timesteps")
    print(f"Val  : {len(df_val)} timesteps")
    print(f"Test : {len(df_test)} timesteps")
    
    # Normalize using TRAINING set statistics only (Doc 11 rule)
    train_mean = df_train.mean()
    train_std  = df_train.std().replace(0, 1)  # avoid division by zero
    
    df_train_n = (df_train - train_mean) / train_std
    df_val_n   = (df_val   - train_mean) / train_std
    df_test_n  = (df_test  - train_mean) / train_std
    
    # Fill any remaining NaNs with 0 (mean after normalization)
    df_train_n = df_train_n.fillna(0)
    df_val_n   = df_val_n.fillna(0)
    df_test_n  = df_test_n.fillna(0)
    
    return df_train_n, df_val_n, df_test_n, train_mean, train_std

df_train_n, df_val_n, df_test_n, train_mean, train_std = prepare_data(df_all)

# Save normalization stats for later use in EnKF module
train_mean.to_csv(MODEL_DIR / "normalization_mean.csv")
train_std.to_csv(MODEL_DIR  / "normalization_std.csv")
print("Normalization stats saved!")

Train: 240 timesteps
Val  : 0 timesteps
Test : 0 timesteps
Normalization stats saved!


In [6]:
class WindSequenceDataset(Dataset):
    def __init__(self, df, in_len=IN_LEN, out_len=OUT_LEN):
        self.data       = torch.FloatTensor(df.values)
        self.in_len     = in_len
        self.out_len    = out_len
        self.target_col = df.columns.get_loc("ws10m_q050")
        
        # Guard against empty datasets
        n = len(df) - in_len - out_len + 1
        self.n_samples = max(0, n)
        
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.in_len]
        y = self.data[idx + self.in_len :
                      idx + self.in_len + self.out_len,
                      self.target_col]
        return x, y

# Build datasets
train_ds = WindSequenceDataset(df_train_n)
val_ds   = WindSequenceDataset(df_val_n)
test_ds  = WindSequenceDataset(df_test_n)

# Only build loaders for non-empty datasets
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False) if len(val_ds)   > 0 else None
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False) if len(test_ds)  > 0 else None

print(f"Train sequences: {len(train_ds)}")
print(f"Val sequences  : {len(val_ds)}")
print(f"Test sequences : {len(test_ds)}")

# Verify shapes
x_sample, y_sample = train_ds[0]
print(f"\nInput shape  : {x_sample.shape}  (seq_len, n_features)")
print(f"Target shape : {y_sample.shape}  (forecast_horizon)")

Train sequences: 232
Val sequences  : 0
Test sequences : 0

Input shape  : torch.Size([5, 66])  (seq_len, n_features)
Target shape : torch.Size([4])  (forecast_horizon)


In [7]:
class TrajGRU(nn.Module):
    """
    Trajectory GRU for wind nowcasting.
    
    Adapted from Doc 8 (TrajGRU nowcasting) but applied to
    tabular wind feature sequences instead of radar images.
    
    Architecture:
      Input → GRU layers → Fully connected → Wind forecast
      
    For a PyTorch beginner: think of GRU as a "memory" layer
    that learns patterns across time steps. It's like the model
    remembers "last 3 hours were gusty, so next hour probably
    still gusty" — but learned automatically from data.
    """
    def __init__(self, n_features, hidden_size=128,
                 n_layers=2, out_len=OUT_LEN, dropout=0.2):
        super(TrajGRU, self).__init__()
        
        self.hidden_size = hidden_size
        self.n_layers    = n_layers
        
        # GRU: processes the sequence and learns temporal patterns
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,      # input shape: (batch, seq, features)
            dropout=dropout,
            bidirectional=False    # only look at past, not future
        )
        
        # Batch normalization: stabilizes training
        self.bn = nn.BatchNorm1d(hidden_size)
        
        # Output layers: compress GRU output to wind forecast
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, out_len)   # one output per forecast step
        )
        
    def forward(self, x):
        # x shape: (batch, seq_len, n_features)
        
        # Run through GRU — output shape: (batch, seq_len, hidden_size)
        gru_out, _ = self.gru(x)
        
        # Take only the last timestep's hidden state
        last_hidden = gru_out[:, -1, :]    # (batch, hidden_size)
        
        # Batch norm
        last_hidden = self.bn(last_hidden)
        
        # Forecast
        forecast = self.fc(last_hidden)    # (batch, out_len)
        
        return forecast

# Build model and move to device (GPU if available)
model = TrajGRU(
    n_features=N_FEATURES,
    hidden_size=128,
    n_layers=2,
    out_len=OUT_LEN,
    dropout=0.2
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model built successfully!")
print(f"Trainable parameters: {n_params:,}")
print(f"\nModel architecture:")
print(model)

Model built successfully!
Trainable parameters: 183,108

Model architecture:
TrajGRU(
  (gru): GRU(66, 128, num_layers=2, batch_first=True, dropout=0.2)
  (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=4, bias=True)
  )
)
